In [ ]:
# BLOQUE 1 — Configuración del descubrimiento del catálogo GTFS

from datetime import datetime, timezone
import uuid

catalog = "workspace"
schema = "gtfs_bogota"

# Estas rutas pertenecen al plano de control: describen qué feeds existen,
# pero no descargan todavía ningún ZIP hacia la capa Raw.
portal_base_url = "https://gis.transmilenio.gov.co/geoportal"
gtfs_group_id = "ca6e3d0acf57461d91228659c1b0d2dd"
catalog_endpoint = (
    f"{portal_base_url}/sharing/rest/content/groups/"
    f"{gtfs_group_id}/search"
)

control_table = f"{catalog}.{schema}.control_gtfs_static_catalog"
observation_table = f"{catalog}.{schema}.audit_gtfs_static_catalog_observations"
catalog_run_id = str(uuid.uuid4())
discovered_at_utc = datetime.now(timezone.utc)

print(f"Endpoint del catálogo: {catalog_endpoint}")
print(f"Corrida de descubrimiento: {catalog_run_id}")

In [ ]:
# BLOQUE 2 — Consulta paginada del catálogo público de TransMilenio

import requests

# El catálogo ArcGIS puede paginar resultados; recorrerlo evita omitir
# versiones antiguas que serán necesarias durante el backfill histórico.
catalog_items = []
next_start = 1

while next_start != -1:
    response = requests.get(
        catalog_endpoint,
        params={
            "f": "json",
            "num": 100,
            "start": next_start,
            "sortField": "modified",
            "sortOrder": "desc",
        },
        timeout=(10, 60),
    )
    response.raise_for_status()

    payload = response.json()
    if "error" in payload:
        raise RuntimeError(f"ArcGIS devolvió un error: {payload['error']}")

    page_results = payload.get("results", [])
    catalog_items.extend(page_results)
    next_start = payload.get("nextStart", -1)

    print(
        f"Página desde {next_start if next_start != -1 else 'final'}: "
        f"{len(page_results)} elementos recibidos"
    )

if not catalog_items:
    raise RuntimeError("El catálogo respondió sin elementos para el grupo GTFS.")

print(f"Elementos totales descubiertos: {len(catalog_items)}")

In [ ]:
# BLOQUE 3 — Normalización y validación del contrato de versiones GTFS

import hashlib
import re

from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# La fecha se toma del título publicado por el proveedor, no de la fecha
# en que nuestro pipeline lo descubrió. Ambas fechas quedan registradas.
version_pattern = re.compile(
    r"GTFS\s+Est[áa]ticos\s+(\d{4}-\d{2}-\d{2})",
    re.IGNORECASE,
)

def arcgis_epoch_to_utc(epoch_ms):
    if epoch_ms is None:
        return None
    return datetime.fromtimestamp(epoch_ms / 1000, timezone.utc)


catalog_rows = []

for item in catalog_items:
    title = item.get("title", "")
    version_match = version_pattern.search(title)

    # Solo promovemos al control los ítems que el proveedor identifica
    # explícitamente como una versión GTFS fechada.
    if not version_match:
        continue

    source_url = item.get("url")
    is_direct_zip_url = bool(
        source_url and source_url.split("?", 1)[0].lower().endswith(".zip")
    )
    metadata_fingerprint = hashlib.sha256(
        "|".join([
            item.get("id", ""),
            title,
            source_url or "",
            str(item.get("modified", "")),
        ]).encode("utf-8")
    ).hexdigest()

    catalog_rows.append((
        version_match.group(1),
        item.get("id"),
        title,
        source_url,
        is_direct_zip_url,
        item.get("type"),
        arcgis_epoch_to_utc(item.get("created")),
        arcgis_epoch_to_utc(item.get("modified")),
        metadata_fingerprint,
        gtfs_group_id,
        catalog_run_id,
        discovered_at_utc,
    ))

if not catalog_rows:
    raise RuntimeError(
        "No se encontraron ítems con el patrón 'GTFS Estáticos AAAA-MM-DD'."
    )

catalog_schema = StructType([
    StructField("feed_version", StringType(), False),
    StructField("portal_item_id", StringType(), False),
    StructField("portal_title", StringType(), False),
    StructField("source_url", StringType(), True),
    StructField("is_direct_zip_url", BooleanType(), False),
    StructField("portal_item_type", StringType(), True),
    StructField("portal_created_at_utc", TimestampType(), True),
    StructField("portal_modified_at_utc", TimestampType(), True),
    StructField("metadata_fingerprint", StringType(), False),
    StructField("source_group_id", StringType(), False),
    StructField("catalog_run_id", StringType(), False),
    StructField("discovered_at_utc", TimestampType(), False),
])

catalog_df = spark.createDataFrame(catalog_rows, schema=catalog_schema)

display(catalog_df.orderBy("feed_version", ascending=False))

In [ ]:
# BLOQUE 4 — Persistencia del catálogo y auditoría de descubrimiento

from delta.tables import DeltaTable

# La tabla de control conserva el último estado conocido de cada ítem del portal.
# El MERGE evita duplicar una versión cuando el Job se ejecuta nuevamente.
if spark.catalog.tableExists(control_table):
    (
        DeltaTable.forName(spark, control_table)
        .alias("target")
        .merge(
            catalog_df.alias("source"),
            "target.portal_item_id = source.portal_item_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        catalog_df
        .write
        .format("delta")
        .saveAsTable(control_table)
    )

# La auditoría es append-only: conserva qué observó cada corrida del Job.
(
    catalog_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(observation_table)
)

print(f"Tabla de control: {control_table}")
print(f"Tabla de auditoría: {observation_table}")

In [ ]:
# BLOQUE 5 — Plan de ingesta para versiones aún no cargadas en Bronze

from pyspark.sql import functions as F

bronze_routes_table = f"{catalog}.{schema}.bronze_gtfs_routes"

# Una versión se considera cargada solo si ya originó registros Bronze.
# Una URL no resuelta nunca se promueve automáticamente a descarga.
loaded_versions_df = (
    spark.table(bronze_routes_table)
    .select("source_feed_version")
    .distinct()
)

ingestion_plan_df = (
    spark.table(control_table)
    .filter(F.col("is_direct_zip_url"))
    .join(
        loaded_versions_df,
        F.col("feed_version") == F.col("source_feed_version"),
        "left_anti",
    )
    .select(
        "feed_version",
        "source_url",
        "portal_title",
        "portal_modified_at_utc",
    )
    .orderBy(F.col("feed_version").desc())
)

display(ingestion_plan_df)

# El primer registro será el candidato del modo latest_uningested.
# Un Workflow posterior recibirá esta salida y lanzará Raw → Staging → Bronze.